<a id="top"></a>

# Lab L0604: validate & translate an MCP tool spec

```yaml
title: "Lab L0604: validate & translate an MCP tool spec"
keywords: mcp, tool spec, inputSchema, input_schema, translate, validate, json schema, l05 design, offline, lab
estimated duration: 30
```

> **Lesson:** L06. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L06/objectives.md). Solutions: `L0604_lab_solutions.ipynb`. **Pure Python (`json`) — no API key, no `mcp` package** (you work with the tool *spec*, the JSON that crosses the wire, exactly like the [L0603 demo](L0603_lecture.ipynb)).
>
> **Why no live server?** The `mcp` package isn't installed in this course env. The conceptual core of MCP — *a well-designed tool serialized to a portable spec* — is fully exercisable offline, which is what this lab does.
>
> **After this lab you can:** translate an [L05](../L05/L0502_lecture.md)-style inline tool definition to an MCP tool spec and back, and validate that a spec has the structure a client needs to discover it.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Translate inline -> MCP tool spec](#2-problem-1--translate-inline---mcp-tool-spec)
- [3. Problem 2 — Prove the design surface survives](#3-problem-2--prove-the-design-surface-survives)
- [4. Problem 3 — Round-trip back to inline](#4-problem-3--round-trip-back-to-inline)
- [5. Problem 4 — Validate a discovered spec](#5-problem-4--validate-a-discovered-spec)
- [6. Problem 5 — Why is the description load-bearing? (written)](#6-problem-5--why-is-the-description-load-bearing-written)
- [7. Self-check](#7-self-check)

## 1. Setup

`json` from the standard library and the [L05](../L05/L0502_lecture.md) `book_meeting` tool as an **inline** Anthropic definition (key: `input_schema`). You will re-package it — not redesign it.

In [ ]:
import json

BOOK_MEETING_INLINE: dict[str, object] = {
    "name": "book_meeting",
    "description": (
        "Book a meeting on the user's calendar. Use this when the user asks to "
        "schedule, set up, or book a meeting with a named attendee at a given time."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "attendee": {"type": "string", "description": "Attendee email, e.g. 'a@b.com'."},
            "start": {"type": "string", "description": "ISO 8601 start, e.g. '2026-06-16T14:00'."},
            "duration_minutes": {"type": "integer", "description": "Minutes, 15-240."},
        },
        "required": ["attendee", "start"],
    },
}


print("inline definition ready for:", BOOK_MEETING_INLINE["name"])

[↑ Back to top](#top)

## 2. Problem 1 — Translate inline -> MCP tool spec

Implement `inline_to_mcp_spec(inline)` returning a dict with keys `name`, `description`, and `inputSchema` (**camelCase**). Carry `name` and `description` over unchanged; the value of `inputSchema` is the inline definition's `input_schema` **unchanged**. The translation is a single key rename — the JSON Schema contents do not change.

In [ ]:
def inline_to_mcp_spec(inline: dict[str, object]) -> dict[str, object]:
    # TODO: return {name, description, inputSchema} — rename input_schema -> inputSchema,
    #       carrying name/description/schema-contents over unchanged.
    raise NotImplementedError("your code here")


mcp_spec = inline_to_mcp_spec(BOOK_MEETING_INLINE)
print(json.dumps(mcp_spec, indent=2))

[↑ Back to top](#top)

## 3. Problem 2 — Prove the design surface survives

Assert that the translation lost nothing: the `inputSchema` equals the original `input_schema`, and `name`/`description` are unchanged. This is the [lecture](L0602_lecture.md) claim (slide 2.2) — *MCP changes packaging, not design* — checked mechanically.

In [ ]:
# TODO: assert mcp_spec['inputSchema'] equals the original input_schema, and that
#       name and description are unchanged. Print a confirmation.
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 4. Problem 3 — Round-trip back to inline

Implement `mcp_spec_to_inline(spec)` reversing the rename (`inputSchema` -> `input_schema`). Round-tripping inline -> MCP -> inline must return the **original** definition exactly — proving no design information is lost in either direction, which is *why* the tool is portable across clients.

In [ ]:
def mcp_spec_to_inline(spec: dict[str, object]) -> dict[str, object]:
    # TODO: reverse the rename: inputSchema -> input_schema, name/description unchanged.
    raise NotImplementedError("your code here")


round_tripped = mcp_spec_to_inline(mcp_spec)
assert round_tripped == BOOK_MEETING_INLINE
print("round-trip inline -> MCP -> inline is lossless")

[↑ Back to top](#top)

## 5. Problem 4 — Validate a discovered spec

A client discovering a server gets back tool specs it did not write — so it should **validate** their structure before trusting them. Implement `validate_spec(spec)` that returns `"OK"` for a well-formed spec, or a `"REJECTED: ..."` string naming the first problem. Require: a non-empty `name`, a non-empty `description`, an `inputSchema` whose `type` is `"object"`. Run it over the four crafted specs (one good, three broken).

In [ ]:
CANDIDATES: dict[str, dict[str, object]] = {
    "good": mcp_spec,
    "no_name": {"name": "", "description": "d", "inputSchema": {"type": "object"}},
    "no_desc": {"name": "t", "description": "", "inputSchema": {"type": "object"}},
    "bad_schema": {"name": "t", "description": "d", "inputSchema": {"type": "string"}},
}


def validate_spec(spec: dict[str, object]) -> str:
    # TODO: reject empty name; reject empty description; reject inputSchema whose
    #       'type' is not 'object'. Otherwise return 'OK'.
    raise NotImplementedError("your code here")


for name, candidate in CANDIDATES.items():
    print(f"{name:11} -> {validate_spec(candidate)}")

[↑ Back to top](#top)

## 6. Problem 5 — Why is the description load-bearing? (written)

Problem 4 rejects a spec with an empty `description`. In a sentence or two: why is a tool spec's `description` *more* consequential for an MCP server than for an inline tool — i.e. who reads it, and when? (Hint: [lecture](L0602_lecture.md) slide 2.3.)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 7. Self-check

Compare against `L0604_lab_solutions.ipynb`. Done when: the inline->MCP translation renames only `input_schema`->`inputSchema`; the round-trip returns the original exactly; and `validate_spec` returns `OK` for `good` and a useful `REJECTED` for `no_name`, `no_desc`, and `bad_schema`. You can say why the published description is runtime system prompt for every connecting model.

[↑ Back to top](#top)